# Risk Deep Dive

**Purpose**: Detailed risk analysis for data science / actuarial review

**Run via**: `make run-notebooks MODE=nyc` or `make all-both`

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Configuration from Makefile
RUN_DIR = os.environ.get("CITIBIKE_RUN_DIR")
mode = os.environ.get("CITIBIKE_MODE", "unknown")

if not RUN_DIR:
    raise ValueError("CITIBIKE_RUN_DIR not set.\nRun via Makefile: make run-notebooks MODE=nyc")

RUN_DIR = Path(RUN_DIR)
run_label = RUN_DIR.name

print(f"Mode: {mode}")
print(f"Run directory: {RUN_DIR}")
print(f"Run label: {run_label}")

# Load CSVs
df_year = pd.read_csv(RUN_DIR / "citibike_trips_by_year.csv")
df_month = pd.read_csv(RUN_DIR / "citibike_trips_by_month.csv")

# Scorecard (find any available radius)
scorecard_files = list(RUN_DIR.glob("axa_partner_scorecard_*m.csv"))
if scorecard_files:
    df_score = pd.read_csv(scorecard_files[0])
    print(f"  Loaded scorecard: {scorecard_files[0].name}")
else:
    df_score = None
    print("  No scorecard files found in RUN_DIR")

print(f"\nLoaded:")
print(f"  df_year:  {len(df_year)} rows")
print(f"  df_month: {len(df_month)} rows")
print(f"  df_score: {len(df_score) if df_score is not None else 'None'} rows")

if df_score is not None:
    print(f"\nScorecard columns: {list(df_score.columns)}")
    if "year" in df_score.columns:
        print(f"Years: {sorted(df_score['year'].dropna().unique())}")
    if "month" in df_score.columns:
        print(f"Months: {sorted(df_score['month'].dropna().unique())}")

# Figure output
FIG_DIR = RUN_DIR.parent.parent / "reports" / run_label / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved: {path}")

In [ ]:
# --- Risk vs Exposure: Multi-panel analysis ---
import math

if df_score is None or len(df_score) == 0:
    print("No scorecard data — skipping panel plots")
else:
    # Prepare data
    plot_df = df_score.copy()

    # Normalize columns
    plot_df = plot_df.rename(columns={"exposure_trips": "trips"}, errors="ignore")

    exposure_col = "trips"
    rate_col = "eb_risk_rate_per_100k_trips"
    crash_col = "crash_count" if "crash_count" in plot_df.columns else None

    # Filter to credible
    if "credibility_flag" in plot_df.columns:
        plot_df = plot_df[plot_df["credibility_flag"] == "credible"].copy()

    # Clean types
    plot_df[exposure_col] = pd.to_numeric(plot_df[exposure_col], errors="coerce")
    plot_df[rate_col] = pd.to_numeric(plot_df[rate_col], errors="coerce")
    if crash_col:
        plot_df[crash_col] = pd.to_numeric(plot_df[crash_col], errors="coerce").fillna(0)

    plot_df = plot_df.dropna(subset=[exposure_col, rate_col])
    plot_df = plot_df[plot_df[exposure_col] > 0].copy()

    # Check if we have any data left
    if len(plot_df) == 0:
        print(f"No credible data for mode={mode} — skipping panel plots")
    else:
        has_period_cols = "year" in plot_df.columns and "month" in plot_df.columns

        # Helper functions
        def draw_scatter(ax, df_slice):
            if crash_col:
                return ax.scatter(
                    df_slice[exposure_col],
                    df_slice[rate_col],
                    c=df_slice[crash_col],
                    alpha=0.6,
                    s=30,
                    vmin=vmin,
                    vmax=vmax,
                )
            else:
                ax.scatter(df_slice[exposure_col], df_slice[rate_col], alpha=0.6, s=30)
                return None

        def mark_hotspots(ax, df_slice):
            if "prevention_hotspot" in df_slice.columns:
                hotspots = df_slice[df_slice["prevention_hotspot"] == True]
                if len(hotspots) > 0:
                    ax.scatter(
                        hotspots[exposure_col],
                        hotspots[rate_col],
                        color="red",
                        s=130,
                        marker="*",
                        edgecolors="black",
                        linewidths=1,
                        label="Prevention Hotspots",
                        zorder=5,
                    )
                    ax.legend(loc="best", fontsize=7)

        def style_axes(ax, title):
            ax.set_xlabel("Exposure (trips)", fontsize=9)
            ax.set_ylabel("EB Risk Rate (per 100k trips)", fontsize=9)
            ax.set_title(title, fontsize=10, fontweight="bold")
            ax.set_xscale("log")
            ax.grid(alpha=0.3)

        # Global color scale
        vmin = vmax = None
        if crash_col and len(plot_df) > 0:
            vmin, vmax = float(plot_df[crash_col].min()), float(plot_df[crash_col].max())

        if not has_period_cols:
            # Overall plot only
            print(f"No year/month columns for mode={mode}. Making overall plot only.")

            fig, ax = plt.subplots(figsize=(7.2, 4.8))
            sc = draw_scatter(ax, plot_df)
            mark_hotspots(ax, plot_df)
            style_axes(ax, f"Risk vs Exposure (overall) — mode={mode}")

            if sc is not None:
                plt.colorbar(sc, ax=ax, label="Crash Count")

            savefig(f"risk_vs_exposure_overall_mode{mode}.png")
            plt.show()

        else:
            # Prepare year/month
            plot_df["year"] = pd.to_numeric(plot_df["year"], errors="coerce")
            plot_df["month"] = pd.to_numeric(plot_df["month"], errors="coerce")
            plot_df = plot_df.dropna(subset=["year", "month"])
            plot_df["year"] = plot_df["year"].astype(int)
            plot_df["month"] = plot_df["month"].astype(int)

            years_to_plot = sorted(plot_df["year"].unique())
            months_to_plot = sorted(plot_df["month"].unique())

            # Guard against empty data after filtering
            if not years_to_plot or not months_to_plot:
                print(f"No data after year/month filtering for mode={mode} — skipping panel plots")
            else:
                print(
                    f"[panel] mode={mode} years={years_to_plot} months={months_to_plot} rows={len(plot_df):,}"
                )

                # =========================================================
                # PANEL A: One subplot per (year, month) - paginated
                # =========================================================
                pairs = [(y, m) for y in years_to_plot for m in months_to_plot]
                total = len(pairs)

                PANELS_PER_PAGE = 12
                NCOLS = 3
                NROWS = int(math.ceil(PANELS_PER_PAGE / NCOLS))
                pages = int(math.ceil(total / PANELS_PER_PAGE))

                print(f"[panelA] Total panels={total} -> pages={pages}")

                for page_idx in range(pages):
                    start = page_idx * PANELS_PER_PAGE
                    page_pairs = pairs[start : start + PANELS_PER_PAGE]

                    figA, axesA = plt.subplots(
                        NROWS, NCOLS, figsize=(4.6 * NCOLS, 3.6 * NROWS), squeeze=False
                    )
                    scatter_for_cbar = None

                    for i, (y, m) in enumerate(page_pairs):
                        r, c = divmod(i, NCOLS)
                        ax = axesA[r][c]

                        d = plot_df[(plot_df["year"] == y) & (plot_df["month"] == m)]
                        sc = draw_scatter(ax, d)
                        mark_hotspots(ax, d)
                        style_axes(ax, f"{y}-{m:02d} (mode={mode})")

                        if scatter_for_cbar is None and sc is not None:
                            scatter_for_cbar = sc

                    # Turn off unused axes
                    for j in range(len(page_pairs), NROWS * NCOLS):
                        r, c = divmod(j, NCOLS)
                        axesA[r][c].axis("off")

                    figA.suptitle(
                        f"Risk vs Exposure — mode={mode} (page {page_idx + 1}/{pages})",
                        fontsize=13,
                        fontweight="bold",
                    )
                    figA.tight_layout(rect=[0.0, 0.0, 1.0, 0.93])

                    if scatter_for_cbar is not None:
                        import matplotlib as mpl

                        sm = mpl.cm.ScalarMappable(
                            norm=scatter_for_cbar.norm, cmap=scatter_for_cbar.cmap
                        )
                        sm.set_array([])
                        visible_axes = [
                            ax for ax in axesA.ravel() if ax.get_visible() and ax.has_data()
                        ]
                        cbar = figA.colorbar(sm, ax=visible_axes, fraction=0.035, pad=0.02)
                        cbar.set_label("Crash Count")

                    savefig(f"risk_vs_exposure_panel_periods_mode{mode}_page{page_idx + 1:02d}.png")
                    plt.show()

                # =========================================================
                # PANEL B: Yearly comparison (overlay years) - one subplot per month
                # =========================================================
                n = len(months_to_plot)
                ncols = min(3, n) if n > 1 else 1
                nrows = int(math.ceil(n / ncols))

                figB, axesB = plt.subplots(
                    nrows, ncols, figsize=(4.6 * ncols, 3.6 * nrows), squeeze=False
                )

                # Auto-select years to overlay (endpoints if too many)
                if len(years_to_plot) > 3:
                    years_overlay = [min(years_to_plot), max(years_to_plot)]
                    overlay_tag = f"endpoints_{years_overlay[0]}_{years_overlay[1]}"
                else:
                    years_overlay = years_to_plot
                    overlay_tag = "allYears"

                for i, m in enumerate(months_to_plot):
                    r, c = divmod(i, ncols)
                    ax = axesB[r][c]

                    any_data = False
                    for y in years_overlay:
                        d = plot_df[(plot_df["year"] == y) & (plot_df["month"] == m)]
                        if d.empty:
                            continue
                        any_data = True
                        ax.scatter(d[exposure_col], d[rate_col], alpha=0.5, s=22, label=str(y))
                        mark_hotspots(ax, d)

                    style_axes(ax, f"Year comparison — month {m:02d} (mode={mode})")
                    if any_data:
                        ax.legend(title="Year", fontsize=8, title_fontsize=9, loc="best")
                    else:
                        ax.text(
                            0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center"
                        )
                        ax.set_axis_off()

                # Turn off unused axes
                for j in range(n, nrows * ncols):
                    r, c = divmod(j, ncols)
                    axesB[r][c].axis("off")

                figB.suptitle(
                    f"Yearly comparison (overlay: {overlay_tag}) — mode={mode}",
                    fontsize=13,
                    fontweight="bold",
                )
                figB.tight_layout(rect=[0.0, 0.0, 1.0, 0.93])

                savefig(f"risk_vs_exposure_panel_year_compare_mode{mode}_{overlay_tag}.png")
                plt.show()

In [ ]:
# --- Check if seasonality analysis is possible ---
can_do_seasonality = False

if df_score is None:
    print("=" * 80)
    print("  SEASONALITY ANALYSIS SKIPPED")
    print("=" * 80)
    print("No scorecard data available.")
elif "year" not in df_score.columns or "month" not in df_score.columns:
    print("=" * 80)
    print("  SEASONALITY ANALYSIS SKIPPED")
    print("=" * 80)
    print("No year/month columns in scorecard.")
else:
    if "credibility_flag" in df_score.columns:
        credible = df_score[df_score["credibility_flag"] == "credible"].copy()
    else:
        credible = df_score.copy()

    if len(credible) == 0:
        print("=" * 80)
        print("  SEASONALITY ANALYSIS SKIPPED")
        print("=" * 80)
        print(f"No credible data for {mode} (min_trips threshold not met).")
        print("This is expected for JC mode (smaller system).")
    else:
        can_do_seasonality = True
        print(f"✓ Seasonality analysis available: {len(credible):,} credible rows")

In [ ]:
# --- Seasonality: line chart (median risk by month, per year) ---
if not can_do_seasonality:
    print("Skipped")
else:
    risk_col = (
        "eb_risk_rate_per_100k_trips"
        if "eb_risk_rate_per_100k_trips" in credible.columns
        else "risk_rate_per_100k_trips"
    )

    # Aggregate: median + IQR per (year, month)
    summary = (
        credible.groupby(["year", "month"])[risk_col]
        .agg(median="median", q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75))
        .reset_index()
    )

    display(summary)

    years = sorted(summary["year"].unique())

    plt.figure(figsize=(10, 5))

    for year in years:
        sub = summary[summary["year"] == year].sort_values("month")
        plt.plot(sub["month"], sub["median"], marker="o", linewidth=2, label=str(int(year)))
        plt.fill_between(sub["month"], sub["q25"], sub["q75"], alpha=0.15)

    plt.title(
        f"Risk Seasonality (Median ± IQR) — mode={mode} ({run_label})",
        fontsize=14,
        fontweight="bold",
    )
    plt.xlabel("Month", fontsize=12)
    plt.ylabel("EB Risk Rate (per 100k trips)", fontsize=12)
    plt.xticks(range(1, 13))
    plt.legend(title="Year")
    plt.grid(True, alpha=0.3, linestyle="--")
    plt.tight_layout()

    savefig("03_seasonality_lines.png")
    plt.show()

In [ ]:
# --- Seasonality: heatmap (year × month) ---
if not can_do_seasonality:
    print("Skipped")
else:
    risk_col = (
        "eb_risk_rate_per_100k_trips"
        if "eb_risk_rate_per_100k_trips" in credible.columns
        else "risk_rate_per_100k_trips"
    )

    pivot = credible.groupby(["year", "month"])[risk_col].median().unstack()

    fig, ax = plt.subplots(figsize=(10, 3 + 0.5 * len(pivot)))

    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{int(m):02d}" for m in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(int(y)) for y in pivot.index])

    ax.set_xlabel("Month", fontsize=12)
    ax.set_ylabel("Year", fontsize=12)
    ax.set_title(
        f"Risk Heatmap (Year × Month) — mode={mode} ({run_label})", fontsize=14, fontweight="bold"
    )

    cbar = plt.colorbar(im, ax=ax, fraction=0.03)
    cbar.set_label("Median EB Risk Rate")

    # Add text labels
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.iloc[i, j]
            if not pd.isna(val):
                color = "white" if val > pivot.values[~np.isnan(pivot.values)].mean() else "black"
                ax.text(j, i, f"{val:.0f}", ha="center", va="center", color=color, fontsize=9)

    plt.tight_layout()
    savefig("04_seasonality_heatmap.png")
    plt.show()

In [ ]:
# --- Boxplot: risk distribution by year ---
if not can_do_seasonality:
    print("Skipped")
else:
    risk_col = (
        "eb_risk_rate_per_100k_trips"
        if "eb_risk_rate_per_100k_trips" in credible.columns
        else "risk_rate_per_100k_trips"
    )

    years = sorted(credible["year"].unique())
    data_by_year = [credible[credible["year"] == y][risk_col].dropna().values for y in years]

    fig, ax = plt.subplots(figsize=(8, 5))

    bp = ax.boxplot(data_by_year, tick_labels=[str(int(y)) for y in years], showfliers=False)

    for median in bp["medians"]:
        median.set_color("red")
        median.set_linewidth(2)

    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("EB Risk Rate (per 100k trips)", fontsize=12)
    ax.set_title(
        f"Risk Distribution by Year — mode={mode} ({run_label})", fontsize=14, fontweight="bold"
    )
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    from matplotlib.lines import Line2D

    ax.legend([Line2D([0], [0], color="red", linewidth=2)], ["Median"], loc="best")

    plt.tight_layout()
    savefig("05_distribution_boxplot.png")
    plt.show()

In [ ]:
# --- Summary ---
print("=" * 60)
print(f"RISK ANALYSIS SUMMARY — mode={mode} ({run_label})")
print("=" * 60)

if df_score is None:
    print("No scorecard data available")
else:
    exp_col = "exposure_trips" if "exposure_trips" in df_score.columns else "trips"
    risk_col = (
        "eb_risk_rate_per_100k_trips"
        if "eb_risk_rate_per_100k_trips" in df_score.columns
        else "risk_rate_per_100k_trips"
    )

    print(f"\nTotal rows in scorecard: {len(df_score):,}")

    if "credibility_flag" in df_score.columns:
        n_credible = (df_score["credibility_flag"] == "credible").sum()
        print(f"Credible station-periods: {n_credible:,}")

    print(f"\nExposure (trips):")
    print(f"  Total: {df_score[exp_col].sum():,.0f}")
    print(f"  Median per station-period: {df_score[exp_col].median():,.0f}")

    print(f"\nRisk ({risk_col}):")
    print(f"  Mean: {df_score[risk_col].mean():.2f}")
    print(f"  Median: {df_score[risk_col].median():.2f}")
    print(f"  Std: {df_score[risk_col].std():.2f}")

    if "prevention_hotspot" in df_score.columns:
        n_hotspots = df_score["prevention_hotspot"].sum()
        print(f"\nPrevention hotspots: {n_hotspots:,}")

    if "eb_m_prior_used" in df_score.columns:
        m_val = df_score["eb_m_prior_used"].iloc[0]
        print(f"\nEB prior strength (m): {m_val:,.0f}")

print("\n" + "=" * 60)